In [23]:
import pandas as pd

path_dataset = "../data/raw/dataset_original.json"

#carga del dataset original en formato json
df = pd.read_json(path_dataset)


import pandas as pd
import numpy as np
import os

# Carga del dataset original
df = pd.read_json('../data/raw/dataset_original.json')
df_original_count = len(df)

# Inicializamos la lista para el log
pipeline_log = []

# 1. DEFINICIÓN DE LA FUNCIÓN
def registrar_paso(df, paso_n, descripcion):
    filas = len(df)
    print(f"Paso {paso_n} registrado: {descripcion} ({filas} filas restantes).")
    
    # 🌟 AGREGA ESTA LÍNEA (Guarda el paso actual en la lista)
    pipeline_log.append({"Paso": paso_n, "Descripción": descripcion, "Filas": filas})
    
    # Convertir la lista de registros en DataFrame y guardarlo
    log_df = pd.DataFrame(pipeline_log)
    log_df.to_csv('../logs/pipeline_log.csv', index=False)
    print("¡Archivo logs/pipeline_log.csv actualizado con éxito!")

# 2. EJECUCIÓN
registrar_paso(df, 0, "Carga inicial del dataset")



    

Paso 0 registrado: Carga inicial del dataset (8160 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!


Paso 1: Eliminación de Duplicados

En la inspección inicial se detectó la posibilidad de registros redundantes. Para asegurar que cada observación sea única y no distorsione los promedios (especialmente en monthly_watch_time_mins), procedemos a eliminar filas idénticas



In [24]:
# Eliminación de duplicados
df = df.drop_duplicates()

# Registro en el log
registrar_paso(df, 1, "Eliminación de filas duplicadas")

Paso 1 registrado: Eliminación de filas duplicadas (8034 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!


Paso 2: Normalización de Categorías (Case Sensitivity)

Se observaron múltiples variantes para una misma categoría en subscription_plan (ej: "Básico", "basico", "BASIC") y favorite_genre (ej: "Acción", "ACCIÓN", "accion"). Esta inconsistencia impide un análisis bivariado correcto. Normalizaremos todo a minúsculas y corregiremos etiquetas

In [25]:
# Normalización de Subscription Plan
# Mapeamos variantes detectadas a nombres estándar
mapa_planes = {
    'basico': 'Básico', 'Básico': 'Básico', 'Basic': 'Básico', 'BASIC': 'Básico',
    'estandar': 'Estándar', 'Estándar': 'Estándar', 'Std': 'Estándar', 'STANDARD': 'Estándar',
    'Premium': 'Premium', 'PREMIUM': 'Premium', 'Premiun': 'Premium', 'premium': 'Premium'
}

df['subscription_plan'] = df['subscription_plan'].str.strip().map(mapa_planes)

# Normalización de Favorite Genre
# Llevamos a minúsculas y quitamos espacios para unificar
df['favorite_genre'] = df['favorite_genre'].str.strip().str.capitalize()

# Registro en el log
registrar_paso(df, 2, "Normalización de texto en planes y géneros")

Paso 2 registrado: Normalización de texto en planes y géneros (8034 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!


Paso 3: Filtrado de Valores Inválidos y Outliers Críticos

En la inspección inicial (Notebook 01) se detectaron registros que no representan usuarios reales, sino errores técnicos o "placeholders":

Edad: Valores de -5 (biológicamente imposible)
 y edades de 130 o 150 años, que distorsionan el perfil demográfico
.
Tiempo de visualización: Valores negativos como -120.0
 y valores extremos de 99,999 (que equivalen a casi 70 días de uso continuo en un mes), indicando datos corruptos
.
Tickets de soporte: Registros con -1, lo cual es un error de conteo

In [26]:
# Definición de filtros lógicos basados en la evidencia
# 1. Filtramos edades dentro de un rango humano razonable (0 a 100 años)
# 2. Filtramos tiempos de visualización positivos y menores al límite de error detectado
# 3. Filtramos tickets de soporte no negativos

df = df[
    (df['age'] >= 0) & (df['age'] < 100) &
    (df['monthly_watch_time_mins'] >= 0) & (df['monthly_watch_time_mins'] < 20000) &
    (df['customer_support_tickets'] >= 0)
]

# Registro en el log del impacto de la limpieza
registrar_paso(df, 3, "Filtrado de valores inválidos (edades neg/150, tiempo neg/99999, tickets -1)")

# Actualizamos el archivo CSV en el disco
log_df = pd.DataFrame(pipeline_log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

Paso 3 registrado: Filtrado de valores inválidos (edades neg/150, tiempo neg/99999, tickets -1) (7662 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!


Paso 4: Tratamiento de nulos
Justificación de la Decisión: ¿Por qué la Mediana?
Robustez ante Outliers: Según la Clase 2, la mediana es el valor típico más representativo cuando la distribución es asimétrica o existen outliers
. Como vimos en tu diagnóstico inicial, monthly_watch_time_mins tiene una media (1107.34) muy alejada de la mediana (757.4) debido a valores extremos como 99,999
. Imputar con la media distorsionaría los datos hacia arriba de forma artificial
.
Preservación de la Retención Estructural: Eliminar las filas con nulos (193 registros) reduciría innecesariamente el volumen de tu dataset
. Las fuentes advierten que el uso prematuro de .dropna() es una de las principales causas de pérdida de información útil y puede destruir la representatividad de ciertos grupos
.
Mecanismo de falta: Al imputar, evitamos introducir sesgos que ocurrirían si elimináramos datos que no faltan completamente al azar (mecanismo MAR), permitiendo que esos usuarios sigan contribuyendo con su información en las otras variables

Para una mayor precisión, realizaremos la imputación utilizando la mediana por grupo (según el subscription_plan), técnica sugerida en la Clase 6 para no mezclar comportamientos de distintos segmentos

In [20]:
# Paso 4: Tratamiento de nulos (Missings)
# Justificación: Imputamos con la mediana agrupada por plan para evitar sesgos por outliers 
# y preservar la retención estructural del dataset.

# Calculamos la mediana del tiempo de visualización para cada plan de suscripción
df['monthly_watch_time_mins'] = df.groupby('subscription_plan')['monthly_watch_time_mins'].transform(
    lambda x: x.fillna(x.median())
)

# Registro del paso en el log de transformaciones
registrar_paso(df, 4, "Imputación de nulos en Watch Time usando mediana por plan")

# Guardado del log actualizado
log_df = pd.DataFrame(pipeline_log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

Paso 4 registrado: Imputación de nulos en Watch Time usando mediana por plan (7662 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!


Evidencia (Markdown):
En el diagnóstico inicial y la visualización de mapas de calor, se detectaron múltiples registros para una misma categoría debido a:
Uso inconsistente de mayúsculas/minúsculas (ej. "Argentina" vs "argentina")
.
Mezcla de nombres estándar con códigos de país (ej. "Argentina" vs "ARG")
.
Presencia de caracteres de ruido y espacios ocultos (ej. "Argentina - " o "Chile ")

In [29]:
# 5. Limpieza inicial: eliminamos espacios en blanco extremos y guiones finales
df['country'] = df['country'].str.strip().str.replace(' -', '', regex=False)
df['favorite_genre'] = df['favorite_genre'].str.strip().str.replace(' -', '', regex=False)

# 2. Diccionario de mapeo para estandarizar códigos y variantes de escritura
mapa_paises = {
    'ARG': 'Argentina', 'argentina': 'Argentina', 'Argentina': 'Argentina',
    'CHL': 'Chile', 'chile': 'Chile', 'Chile': 'Chile',
    'URY': 'Uruguay', 'uruguay': 'Uruguay', 'Uruguay': 'Uruguay',
    'PER': 'Perú', 'perú': 'Perú', 'Peru': 'Perú', 'Perú': 'Perú',
    'MEX': 'México', 'méxico': 'México', 'Mexico': 'México', 'México': 'México',
    'COL': 'Colombia', 'colombia': 'Colombia', 'Colombia': 'Colombia',
    'BRA': 'Brasil', 'brazil': 'Brasil', 'Brasil': 'Brasil', 'Brazil': 'Brasil' ,'brasil': 'Brasil'
}

mapa_generos = {
    'Accion': 'Acción', 'ACCION': 'Acción', 'ACCIÓN': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    'Crimen': 'Crime', 'CRIME': 'Crime', 'Crimen': 'Crime',
    'Documentary': 'Documental', 'documental': 'Documental', 'Doc': 'Documental',
    'Comedy': 'Comedia', 'COMEDIA': 'Comedia', 'comedy': 'Comedia',
    'Thriler': 'Thriller', 'thriler': 'Thriller', 'THRILLER': 'Thriller'
}

# 3. Aplicación del mapeo y capitalización estándar
df['country'] = df['country'].replace(mapa_paises)
df['favorite_genre'] = df['favorite_genre'].replace(mapa_generos).str.capitalize()

# 4. Registro en el log (Trazabilidad obligatoria)
registrar_paso(df, 5, "Estandarización de nombres de países y géneros favoritos")

# Guardamos el progreso en el archivo CSV
log_df = pd.DataFrame(pipeline_log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

Paso 5 registrado: Estandarización de nombres de países y géneros favoritos (7662 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!


In [28]:
# Paso 6: Exportación de datos limpios
# Guardamos en formato CSV para facilitar la lectura en las siguientes notebooks
df.to_csv('../data/processed/dataset_final.csv', index=False)

# Guardado definitivo del log de transformaciones
log_df = pd.DataFrame(pipeline_log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

registrar_paso(df, 6, "Exportación de datos limpios")

print("✅ Proceso completado.")
print("1. Dataset procesado guardado en: data/processed/dataset_final.csv")
print("2. Log de trazabilidad actualizado en: logs/pipeline_log.csv")

Paso 6 registrado: Exportación de datos limpios (7662 filas restantes).
¡Archivo logs/pipeline_log.csv actualizado con éxito!
✅ Proceso completado.
1. Dataset procesado guardado en: data/processed/dataset_final.csv
2. Log de trazabilidad actualizado en: logs/pipeline_log.csv
